# SFMTGen — SFMT607 equidistribution

**SFMT** (SIMD-oriented Fast Mersenne Twister, Saito-Matsumoto 2008)
batches state updates into 128-bit super-words and exploits SIMD
shifts. Its characteristic polynomial is **not primitive** — the
state is $128N$ bits but the period exponent is the prime
$\mathrm{MEXP}$ — so the equidistribution test must use the
**`simd_notprimitive`** method (Saito-Matsumoto 2008, §3.2),
which projects onto the largest primitive factor and accounts for
the 4 lanes per 128-bit word.

This notebook uses SFMT607 (the smallest variant). See
[generators/SFMTGen.md](../../generators/SFMTGen.md).


## Imports


In [ ]:
# stamp:auto-generated
from regpoly.core.generator import Generator
from regpoly.core.combination import Combination
from regpoly.core.transformation import Transformation
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest,
    METHOD_MATRICIAL, METHOD_HARASE,
    METHOD_NOTPRIMITIVE, METHOD_SIMD_NOTPRIMITIVE,
)

INT_MAX = 2**31 - 1


## Construct the generator — _Saito & Matsumoto (2008)_


In [ ]:
# SFMT607 parameters (Saito-Matsumoto 2008 — supplementary table).
gen = Generator.create("SFMTGen", L=32,
    mexp=607, pos1=2, sl1=15, sl2=3, sr1=13, sr2=3,
    msk=[
        0xFDFF37FF, 0xEF7F3F7D, 0xFF777B7D, 0x7FF7FB2F,
    ])
print(gen.display())


## Wrap in a `Combination`


In [ ]:
# Wrap the generator in a single-component Combination. The
# Combination is the live object the search loop iterates and the
# equidistribution test consumes.
comb = Combination(J=1, Lmax=gen.L)
comb.components[0].add_gen(gen)
comb.reset()
print(f"k_g = {comb.k_g}, L = {comb.L}")


## Equidistribution test

SFMT's $\chi_f$ is reducible (state has $128N \ne $ period exponent bits). The **`simd_notprimitive`** method (Saito-Matsumoto 2008, §3.2) projects onto the largest primitive factor and accounts for the 4 lanes per 128-bit word.


In [ ]:
# Build the equidistribution test and run it. We cap `delta` at
# INT_MAX so nothing is rejected — we just want the score.
test = EquidistributionTest(
    L=gen.L,
    delta=[INT_MAX] * (gen.L + 1),
    mse=INT_MAX,
    method=METHOD_SIMD_NOTPRIMITIVE,
)
result = test.run(comb)

print(f"SE (Σ gaps)   = {result.se}")
print(f"verified      = {result.verified}")
print(f"first 10 gaps = {[result.ecart[i] for i in range(1, min(11, gen.L + 1))]}")


## Catalog entry

The published version of this parameter set lives in the REGPOLY catalog under `library_id = "sfmt607"`. To load it programmatically without hard-coding parameters:

```python
from regpoly.library import Catalog
cat = Catalog('docs/library')
cat.load()
_, entry = cat.generator('sfmt607')
# entry.components[0] carries the same params as constructed above
```
